In [1]:
import json, os
from pathlib import Path
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

PROJECT = Path(r"C:\Users\shlok\projects\ddp-llm\parser")
DATA = PROJECT / "data"
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
OUTPUT_DIR = PROJECT / "checkpoints" / "qwen3b-qlora-v1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def load_jsonl(path):
    return [json.loads(l) for l in Path(path).read_text(encoding="utf-8").splitlines() if l.strip()]

train_raw = load_jsonl(DATA / "train.jsonl")
test_raw = load_jsonl(DATA / "test.jsonl")
print(f"train: {len(train_raw)}, test: {len(test_raw)}")

train: 1278, test: 210


In [2]:
SYSTEM_PROMPT = """You are a parser that converts a user's natural language response into a subset of the shown options.

The user is shown 4 options labeled A, B, C, D and gives feedback about which are close to what they want. Your job is to output which options the user views favorably.

Output format (JSON only, nothing else):
- A JSON list of the favored labels, e.g. ["A", "B"]
- [] if the user explicitly rejects ALL options ("none of these", "all wrong")
- "*" if the utterance is off-topic OR expresses no usable preference ("I don't know", "they all look the same", "I love football")

Rules:
- Any positive signal about an option means it goes in the list.
- "X is better than Y" endorses only X, not Y.
- "X and Y are both good, X is better" endorses both X and Y.
- Negations like "not D" or "anything but B" mean the remaining options go in the list.
- Questions like "is it A?" are treated as tentative endorsement of A.

Output ONLY the JSON. No explanation, no prose."""

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

train_raw = load_jsonl(DATA / "train.jsonl")
test_raw = load_jsonl(DATA / "test.jsonl")
print(f"train: {len(train_raw)}, test: {len(test_raw)}")

def to_chat(ex):
    user = f'Options: {ex["options"]}\nUser: {ex["utterance"]}'
    assistant = json.dumps(ex["label"])
    return {"messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user},
        {"role": "assistant", "content": assistant},
    ]}

train_ds = Dataset.from_list([to_chat(e) for e in train_raw])
test_ds  = Dataset.from_list([to_chat(e) for e in test_raw])

train: 1278, test: 210


In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="cuda",
)
model = prepare_model_for_kbit_training(model)
print(f"VRAM after load: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

W0811 19:28:24.662000 24940 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


VRAM after load: 2.69 GB


In [4]:
lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


In [5]:
sft_config = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    bf16=False, fp16=False,
    optim="paged_adamw_8bit",
    report_to="none",
    max_length=512,
    packing=False,
    gradient_checkpointing=True,
    max_grad_norm=1.0,
)

trainer = SFTTrainer(
    model=model, args=sft_config,
    train_dataset=train_ds, eval_dataset=test_ds,
    processing_class=tokenizer,
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Tokenizing train dataset:   0%|          | 0/1278 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1278 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/210 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/210 [00:00<?, ? examples/s]

In [6]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.060726,0.059331,0.065741,358706.000000,0.987113
2,0.051754,0.054380,0.054851,717412.000000,0.987940
3,0.047943,0.053968,0.048771,1076118.000000,0.988137


TrainOutput(global_step=240, training_loss=0.18134371017416318, metrics={'train_runtime': 1989.9982, 'train_samples_per_second': 1.927, 'train_steps_per_second': 0.121, 'total_flos': 1.8301691353350144e+16, 'train_loss': 0.18134371017416318, 'epoch': 3.0})

In [7]:
import gc
try:
    del trainer, model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM after cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

VRAM after cleanup: 1.26 GB


In [8]:
from peft import PeftModel

# sanity check what checkpoint dir got saved
print("checkpoints:", os.listdir(OUTPUT_DIR))

base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="cuda",
)

# adjust checkpoint number based on what you see above (should be 219 for 1279 train / 16 eff batch * 3 epochs)
CHECKPOINT = "checkpoint-240"
adapter_path = str(OUTPUT_DIR / CHECKPOINT)
ft_model = PeftModel.from_pretrained(base, adapter_path)
ft_model.eval()
print("adapter loaded")

checkpoints: ['checkpoint-160', 'checkpoint-240', 'checkpoint-80', 'README.md']


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


adapter loaded


In [9]:
import os
print(os.listdir(OUTPUT_DIR))

['checkpoint-160', 'checkpoint-240', 'checkpoint-80', 'README.md']


In [10]:
from tqdm.auto import tqdm
from collections import defaultdict

def generate_ft(messages, max_new_tokens=40):
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to("cuda")
    with torch.no_grad():
        out = ft_model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()

def parse_output(text):
    text = text.strip()
    if text.startswith("```"):
        text = text.strip("`").lstrip("json").strip()
    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        return None
    if parsed == "*" or (isinstance(parsed, list) and all(x in ["A","B","C","D"] for x in parsed)):
        return parsed
    return None

def labels_equal(a, b):
    if a == "*" or b == "*":
        return a == b
    if isinstance(a, list) and isinstance(b, list):
        return set(a) == set(b)
    return False

valid, correct = 0, 0
by_cat = defaultdict(lambda: [0, 0])  # [correct, total]
failures = []

for ex in tqdm(test_raw):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f'Options: {ex["options"]}\nUser: {ex["utterance"]}'},
    ]
    raw = generate_ft(messages)
    parsed = parse_output(raw)
    cat = ex.get("category", "unknown")
    by_cat[cat][1] += 1
    if parsed is not None:
        valid += 1
    if parsed is not None and labels_equal(parsed, ex["label"]):
        correct += 1
        by_cat[cat][0] += 1
    else:
        failures.append({"utterance": ex["utterance"], "gold": ex["label"], "parsed": parsed, "raw": raw, "category": cat})

n = len(test_raw)
print(f"\nformat validity: {valid}/{n} = {100*valid/n:.1f}%")
print(f"exact match:     {correct}/{n} = {100*correct/n:.1f}%\n")
print("per category:")
for c in sorted(by_cat):
    ok, tot = by_cat[c]
    print(f"  {c}: {ok}/{tot} = {100*ok/tot:.1f}%")

  0%|          | 0/210 [00:00<?, ?it/s]

C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



format validity: 210/210 = 100.0%
exact match:     209/210 = 99.5%

per category:
  comparative: 26/26 = 100.0%
  mixed_sentiment: 27/28 = 96.4%
  multi_positive: 13/13 = 100.0%
  negation: 61/61 = 100.0%
  off_topic: 22/22 = 100.0%
  reject_all: 29/29 = 100.0%
  single_positive: 9/9 = 100.0%
  uncertainty: 22/22 = 100.0%


In [11]:
print("=== FAILURES ===")
for f in failures:
    print(f"[{f['category']}] {f['utterance']!r}")
    print(f"   gold:   {f['gold']}")
    print(f"   parsed: {f['parsed']}")
    print(f"   raw:    {f['raw']!r}")
    print()

=== FAILURES ===
[mixed_sentiment] 'A perfect, B terrible, C okay, D terrible'
   gold:   ['A', 'C']
   parsed: ['A']
   raw:    '["A"]'



In [12]:
import json
from pathlib import Path

train = [json.loads(l) for l in open(PROJECT / "data" / "train.jsonl", encoding="utf-8")]
test = [json.loads(l) for l in open(PROJECT / "data" / "test.jsonl", encoding="utf-8")]

train_utts = {e["utterance"].lower().strip() for e in train}
test_utts = {e["utterance"].lower().strip() for e in test}
overlap = train_utts & test_utts
print(f"train: {len(train_utts)}, test: {len(test_utts)}, overlap: {len(overlap)}")
if overlap:
    for u in list(overlap)[:10]:
        print(f"  {u!r}")

train: 1278, test: 210, overlap: 0


In [13]:
MY_UTTERANCE = "eh, only B feels right"
OPTIONS = ["A", "B", "C", "D"]

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": f'Options: {OPTIONS}\nUser: {MY_UTTERANCE}'},
]
raw = generate_ft(messages)
parsed = parse_output(raw)
print(f"utterance: {MY_UTTERANCE!r}")
print(f"raw:       {raw!r}")
print(f"parsed:    {parsed}")

utterance: 'eh, only B feels right'
raw:       '["B"]'
parsed:    ['B']


In [14]:
test_utterances = [
    "eh, only B feels right",
    "throw out everything but the third one",
    "hard pass on all four",
    "A? maybe. dunno.",
    "the last two are garbage",
    "B leaves D in the dust",
    "A is as good as C but not better than D",
    "meh whatever pick anything",
    "the second and fourth ones look promising",
    "not really feeling any of them tbh",
    "A is decent, B slightly better, C and D no",
    "give me option C or nothing",
]

for u in test_utterances:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f'Options: {["A","B","C","D"]}\nUser: {u}'},
    ]
    raw = generate_ft(messages)
    parsed = parse_output(raw)
    print(f"{u!r:60s} -> {parsed}")

'eh, only B feels right'                                     -> ['B']
'throw out everything but the third one'                     -> ['C']
'hard pass on all four'                                      -> []
'A? maybe. dunno.'                                           -> *
'the last two are garbage'                                   -> ['A']
'B leaves D in the dust'                                     -> ['B']
'A is as good as C but not better than D'                    -> ['A', 'C']
'meh whatever pick anything'                                 -> *
'the second and fourth ones look promising'                  -> ['B', 'D']
'not really feeling any of them tbh'                         -> []
'A is decent, B slightly better, C and D no'                 -> ['A', 'B']
'give me option C or nothing'                                -> ['C']


In [15]:
MY_UTTERANCE = "E is good"
OPTIONS = ["A", "B", "C", "D","E"]

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": f'Options: {OPTIONS}\nUser: {MY_UTTERANCE}'},
]
raw = generate_ft(messages)
parsed = parse_output(raw)
print(f"utterance: {MY_UTTERANCE!r}")
print(f"raw:       {raw!r}")
print(f"parsed:    {parsed}")

utterance: 'E is good'
raw:       '["E"]'
parsed:    None


In [16]:
def parse_output(text, valid_letters=None):
    if valid_letters is None:
        valid_letters = ["A","B","C","D"]
    text = text.strip()
    if text.startswith("```"):
        text = text.strip("`").lstrip("json").strip()
    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        return None
    if parsed == "*" or (isinstance(parsed, list) and all(x in valid_letters for x in parsed)):
        return parsed
    return None

In [17]:
tests = [
    ("E is good", ["A","B","C","D","E"]),
    ("A and E are best", ["A","B","C","D","E"]),
    ("everything except E is wrong", ["A","B","C","D","E"]),
    ("the fifth one is good", ["A","B","C","D","E"]),
    ("not E, not D", ["A","B","C","D","E"]),
    ("F is the best", ["A","B","C","D","E","F"]),
    ("the last one only", ["A","B","C","D","E","F","G"]),
    ("none of these", ["A","B","C","D","E","F"]),
    ("I don't know", ["A","B","C","D","E","F"]),
    ("only G matters", ["A","B","C","D","E","F","G"]),
    ("the last one only", ["A","B","C","D","E","F","G","H","I","J"]),
    ("the last one is out for sure", ["A","B","C","D","E","F","G","H","I","J"]),
]

for utt, opts in tests:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f'Options: {opts}\nUser: {utt}'},
    ]
    raw = generate_ft(messages)
    parsed = parse_output(raw, valid_letters=opts)
    print(f"{utt!r:50s} opts={len(opts)} -> raw={raw!r:20s} parsed={parsed}")

'E is good'                                        opts=5 -> raw='["E"]'              parsed=['E']
'A and E are best'                                 opts=5 -> raw='["A", "E"]'         parsed=['A', 'E']
'everything except E is wrong'                     opts=5 -> raw='["E"]'              parsed=['E']
'the fifth one is good'                            opts=5 -> raw='["E"]'              parsed=['E']
'not E, not D'                                     opts=5 -> raw='["A", "B", "C"]'    parsed=['A', 'B', 'C']
'F is the best'                                    opts=6 -> raw='["F"]'              parsed=['F']
'the last one only'                                opts=7 -> raw='["E", "F", "G"]'    parsed=['E', 'F', 'G']
'none of these'                                    opts=6 -> raw='[]'                 parsed=[]
"I don't know"                                     opts=6 -> raw='"*"'                parsed=*
'only G matters'                                   opts=7 -> raw='["G"]'              parse

In [18]:
MY_UTTERANCE = "Cant tell really but A is better than the rest"
OPTIONS = ["A", "B", "C", "D","E"]

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": f'Options: {OPTIONS}\nUser: {MY_UTTERANCE}'},
]
raw = generate_ft(messages)
parsed = parse_output(raw)
print(f"utterance: {MY_UTTERANCE!r}")
print(f"raw:       {raw!r}")
print(f"parsed:    {parsed}")

utterance: 'Cant tell really but A is better than the rest'
raw:       '["A"]'
parsed:    ['A']


In [19]:
# find failures
failures = []
for i, ex in enumerate(test_raw):
    # you'll need to reconstruct which examples failed — this depends on how the eval loop stored results
    # if you have a list of parsed outputs or predictions, use that
    pass

In [20]:
utt = "is it A?"
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": f'Options: ["A", "B", "C", "D"]\nUser: {utt}'},
]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to("cuda")
with torch.no_grad():
    out = ft_model.generate(**inputs, max_new_tokens=40, do_sample=False, pad_token_id=tokenizer.eos_token_id)
print(repr(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)))

'["A"]'


In [21]:
novel_tests = [
    ("eh, only B feels right", ["B"]),
    ("throw out everything but the third one", ["C"]),
    ("hard pass on all four", []),
    ("A? maybe. dunno.", None),  # ambiguous — accept [A] or *
    ("the last two are garbage", ["A","B"]),
    ("B leaves D in the dust", ["B"]),
    ("meh whatever pick anything", "*"),
    ("the second and fourth ones look promising", ["B","D"]),
    ("A is decent, B slightly better, C and D no", ["A","B"]),
    ("give me option C or nothing", ["C"]),
]

correct = 0
print(f"{'utterance':<50s} {'expected':<20s} {'got':<20s} {'result'}")
print("-" * 100)
for utt, expected in novel_tests:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f'Options: ["A", "B", "C", "D"]\nUser: {utt}'},
    ]
    raw = generate_ft(messages)
    got = parse_output(raw)
    if expected is None:
        result = "ambig"
    elif got == expected or (isinstance(got, list) and isinstance(expected, list) and set(got) == set(expected)):
        result = "✓"
        correct += 1
    else:
        result = "✗"
    print(f"{utt[:48]:<50s} {str(expected)[:18]:<20s} {str(got)[:18]:<20s} {result}")

print(f"\ncorrect (excluding ambiguous): {correct}/9")

utterance                                          expected             got                  result
----------------------------------------------------------------------------------------------------
eh, only B feels right                             ['B']                ['B']                ✓
throw out everything but the third one             ['C']                ['C']                ✓
hard pass on all four                              []                   []                   ✓
A? maybe. dunno.                                   None                 *                    ambig
the last two are garbage                           ['A', 'B']           ['A']                ✗
B leaves D in the dust                             ['B']                ['B']                ✓
meh whatever pick anything                         *                    *                    ✓
the second and fourth ones look promising          ['B', 'D']           ['B', 'D']           ✓
A is decent, B slightly better, C a

In [22]:
# OOD generalization test — 1.5B v1.5
import json, re

def parse_output(text, valid_letters=None):
    if valid_letters is None:
        valid_letters = ["A", "B", "C", "D"]
    text = text.strip()
    if text.startswith("```"):
        text = text.strip("`").lstrip("json").strip()
    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        return None
    if parsed == "*":
        return "*"
    if isinstance(parsed, list) and all(x in valid_letters for x in parsed):
        return parsed
    return None

def generate_ft(messages, max_new_tokens=40):
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to("cuda")
    with torch.no_grad():
        out = ft_model.generate(**inputs, max_new_tokens=max_new_tokens,
                                do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# OOD test cases from session 3
tests = [
    # Novel utterances
    (["A","B","C","D"], "eh, only B feels right", ["B"]),
    (["A","B","C","D"], "throw out everything but the third one", ["C"]),
    (["A","B","C","D"], "hard pass on all four", []),
    (["A","B","C","D"], "A? maybe. dunno.", None),  # ambiguous
    (["A","B","C","D"], "the last two are garbage", ["A","B"]),
    (["A","B","C","D"], "B leaves D in the dust", ["B"]),
    (["A","B","C","D"], "meh whatever pick anything", "*"),
    (["A","B","C","D"], "the second and fourth ones look promising", ["B","D"]),
    (["A","B","C","D"], "A is decent, B slightly better, C and D no", ["A","B"]),
    (["A","B","C","D"], "give me option C or nothing", ["C"]),

    # Variable N with unseen letters
    (["A","B","C","D","E"], "E is good", ["E"]),
    (["A","B","C","D","E"], "A and E are best", ["A","E"]),
    (["A","B","C","D","E"], "everything except E is wrong", ["E"]),
    (["A","B","C","D","E"], "the fifth one is good", ["E"]),
    (["A","B","C","D","E"], "not E, not D", ["A","B","C"]),
    (["A","B","C","D","E","F"], "F is the best", ["F"]),
    (["A","B","C","D","E","F","G"], "the last one only", ["G"]),
    (["A","B","C","D","E","F"], "none of these", []),
    (["A","B","C","D","E","F"], "I don't know", "*"),
    (["A","B","C","D","E","F","G"], "only G matters", ["G"]),
]

correct = 0
ambiguous = 0
print(f"{'options':<25s} {'utterance':<50s} {'expected':<20s} {'got':<20s} {'✓/✗'}")
print("-" * 130)
for options, utt, expected in tests:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f'Options: {options}\nUser: {utt}'},
    ]
    raw = generate_ft(messages)
    got = parse_output(raw, valid_letters=options)
    
    if expected is None:
        result = "ambig"
        ambiguous += 1
    elif got == expected or (isinstance(got, list) and isinstance(expected, list) and set(got) == set(expected)):
        result = "✓"
        correct += 1
    else:
        result = "✗"
    
    options_str = str(options)[:23]
    utt_str = utt[:48]
    exp_str = str(expected)[:18]
    got_str = str(got)[:18]
    print(f"{options_str:<25s} {utt_str:<50s} {exp_str:<20s} {got_str:<20s} {result}")

print(f"\ncorrect: {correct}/{len(tests)-ambiguous} (excluding {ambiguous} ambiguous)")

options                   utterance                                          expected             got                  ✓/✗
----------------------------------------------------------------------------------------------------------------------------------
['A', 'B', 'C', 'D']      eh, only B feels right                             ['B']                ['B']                ✓
['A', 'B', 'C', 'D']      throw out everything but the third one             ['C']                ['C']                ✓
['A', 'B', 'C', 'D']      hard pass on all four                              []                   []                   ✓
['A', 'B', 'C', 'D']      A? maybe. dunno.                                   None                 *                    ambig
['A', 'B', 'C', 'D']      the last two are garbage                           ['A', 'B']           ['A']                ✗
['A', 'B', 'C', 'D']      B leaves D in the dust                             ['B']                ['B']                ✓
['A', 'B', 'C', 